In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/xeong_oxx/miniconda3/envs/qlora_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
import os

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    'google/gemma-2-2b-it',
    torch_dtype = torch.bfloat16,
    device_map='auto'   
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 288/288 [00:00<00:00, 429.98it/s, Materializing param=model.norm.weight]                                


In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    'google/gemma-2-2b-it'
)

In [8]:
model.eval()

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [9]:
model.generation_config.max_new_tokens = 2048

In [10]:
model.generation_config

GenerationConfig {
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "eos_token_id": [
    1,
    107
  ],
  "max_new_tokens": 2048,
  "pad_token_id": 0
}

In [11]:
input_text = "What is your name?"

In [12]:
input_ids = tokenizer(input_text, return_tensors='pt').to(model.device)
print(input_ids)

{'input_ids': tensor([[     2,   1841,    603,    861,   1503, 235336]], device='cuda:6'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:6')}


In [13]:
tokenizer.decode([1841,  603,  861, 1503])

'What is your name'

In [14]:
outputs = model.generate(**input_ids)

In [15]:
outputs

tensor([[     2,   1841,    603,    861,   1503, 235336,    109,   2926,   1503,
            603, 137061, 235265,    590,   1144,    671,  16481,  20409, 235265,
         235248,    108,    107]], device='cuda:6')

In [16]:
print(tokenizer.decode(outputs, skip_special_tokens=True))

['What is your name?\n\nMy name is Gemma. I am an AI assistant. \n']


In [17]:
print(tokenizer.decode(outputs, skip_special_tokens=True)[0][len(input_text):].lstrip())

My name is Gemma. I am an AI assistant. 



In [33]:
input_text = '안녕하세요.'

input_ids = tokenizer(input_text,return_tensors='pt').to(model.device)

In [34]:
input_ids

{'input_ids': tensor([[     2, 238179, 243415, 204551, 235265]], device='cuda:6'), 'attention_mask': tensor([[1, 1, 1, 1, 1]], device='cuda:6')}

In [35]:
outputs = model.generate(
    input_ids = input_ids.input_ids,
    return_dict_in_generate = True,
    output_scores = True
)

In [37]:
len(outputs.scores)

29

In [ ]:
# scores = (
#     tensor_stemp0,
#     tensor_stemp1,
#     tensor_stemp2,
#     ...
# )
outputs.scores[1].shape

torch.Size([1, 256000])

In [38]:
# 모델이 생성한 답변    
print(tokenizer.decode(outputs.sequences[0],skip_special_tokens=True)[len(input_text):].lstrip())

저는 한국어로 대화할 수 있는 AI 모델입니다. 😊

무엇을 도와드릴까요? 



In [40]:
len(outputs.sequences[0])

34

In [41]:
len(input_ids.input_ids[0])

5

In [54]:
print(outputs.scores[0][0])

tensor([-8.8125, -5.0312, -4.5312,  ..., -8.2500, -8.1875, -6.8750],
       device='cuda:6')


In [46]:
len(tokenizer)

256000

In [ ]:
for i in range(len(outputs.scores)):
    # output.scores는 logit distribution을 return 하는거고 
    one_logit = outputs.scores[i][0]
    #그곳에서 argmax를 통해 greedy 선택을 함
    predict_id = torch.argmax(one_logit)
    # 그 값을 decode
    print(f"{i}. {tokenizer.decode(predict_id)}")

0.  저
1. 는
2.  한국
3. 어
4. 로
5.  대
6. 화
7. 할
8.  수
9.  있는
10.  AI
11.  모델
12. 입니다
13. .
14.  😊
15. 


16. 무
17. 엇
18. 을
19.  도
20. 와
21. 드
22. 릴
23. 까
24. 요
25. ?
26.  
27. 

28. <end_of_turn>


In [59]:
predict_id = torch.argmax(one_logit)

In [60]:
predict_id

tensor(236214, device='cuda:6')

In [61]:
tokenizer.decode(predict_id)

'는'

# Top-p, Top-k, min_p

In [90]:
text= "세종대왕이 누구야?"
input_ids = tokenizer(text, return_tensors='pt').to(model.device)

In [108]:
# Greedy Search
output = model.generate(
    input_ids = input_ids.input_ids,
    temperature =0.0,
)

In [109]:
output

tensor([[     2, 237533, 238777, 236800, 240292, 235832, 130673, 237302, 238305,
         235336,    109,    688, 237533, 238777, 236800, 240292,    688, 236648,
          42916, 237700,  27941, 236800, 236137, 235248, 235274, 235310, 236800,
         159970,  26291, 235269, 235248, 235274, 235310, 235310, 235304, 237029,
         124431, 235248, 235274, 235310, 235308, 235276, 237029, 109535,  90869,
         237601, 236791, 235248,  88513, 235265, 164305,   5231, 237602, 237700,
         236137, 235248, 235274, 235276, 236800, 159970,    688, 235832, 112778,
         236840,  83133, 236432, 238986, 235269,   5231, 237602, 237700, 236137,
          61169, 236417, 236791, 235248, 242329, 238151, 236214,  52604, 242406,
            688,  26291,  41896, 241157, 236183, 236464,  69581, 235265,    109,
            688, 237533, 238777, 236800, 240292, 236137, 235248, 244258, 236770,
         240198, 142995, 237603,    688, 236648, 115049, 237233,  81673, 236039,
         235265,    109, 235

In [110]:
print(type(tokenizer.decode(output, skip_special_tokens=True)))
print(output.shape)

<class 'list'>
torch.Size([1, 478])


In [111]:
tokenizer.decode(output[0][2])

'종'

In [112]:
print(tokenizer.decode(output,skip_special_tokens=True)[0][len(text):].lstrip())

**세종대왕**은 조선 시대의 14대 왕으로, 1443년부터 1450년까지 재위를 했다. 그는 **조선의 10대 왕**이라고도 불리며, **조선의 역사를 빛내는 영웅**으로 여겨지고 있다.

**세종대왕의 뛰어난 업적**은 다음과 같다.

* **국제적인 관계 개선**: 세종대왕은 국제적인 관계 개선에 힘썼으며, 일본과의 관계를 개선하고, 중국과의 관계를 맺었다.
* **문화 발전**: 세종대왕은 **국가의 문화 발전을 위해 노력**했다. 그는 **한글을 정비하고, 궁궐의 건축을 개선하고, 예술과 학문을 발전시켰다**.
* **정치적 발전**: 세종대왕은 **정치적 발전을 위해 노력**했다. 그는 **정치적 체계를 개선하고, 국민의 권리를 보장했다**.
* **경제적 발전**: 세종대왕은 **경제적 발전을 위해 노력**했다. 그는 **농업과 무역을 발전시켰다**.

**세종대왕의 업적은 조선의 역사를 빛내는 영웅**으로 여겨지며, 그의 업적은 현재까지도 우리에게 큰 영향을 미치고 있다.


**세종대왕의 삶과 업적에 대한 더 자세한 정보는 다음과 같은 자료를 참고할 수 있습니다.**

* **세종대왕의 삶**: https://www.google.com/search?q=세종대왕+의+삶
* **세종대왕의 업적**: https://www.google.com/search?q=세종대왕+의+업적
* **세종대왕의 영향**: https://www.google.com/search?q=세종대왕+의+영향





In [117]:
text= "세종대왕이 누구야?"
input_ids = tokenizer(text, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids = input_ids.input_ids,
    temperature = 3.0,
    do_sample = True,
)

In [119]:
print(tokenizer.decode(output,skip_special_tokens=True)[0][len(text):].lstrip())

다음 자료들은 누님 생부입니다는 이물을 확인하려어 목지했습니다.:
***


다음은 어쩌죠 인상어 및 글쎄 부른 것 같은 내용 이 부를 사용 설명요소 포함 기본 개술

누님인 아닌 것보다 사문화적 측지를 강조 하셨지여 정확하게 명하오라는 조항 있네 주식

제1기원왕
* 유원(李均的): '11일 월/화' 유사이 사마체형식 어떠한 행행 행하거나 자재 유체 조지 후 한계
            * 주말 연의: 부당 엄격 한위치 시 주의 상소원 주체 (위 주소 제한 가능 해마다 연속이 있는 이 사전 유상 협과로 구성이 특전한 경우가 많애요),
나나, 그 후 그런 행기가 보니 유의유주인. "세속주의 전 왜곡 사인": 일의 유해로 분화됨 -> 어쩔 방법은 다섯번 재발제

- 제21차 정권, 곧 정전일 조세는 유도가 가능 해

###  제로기념을 위해 제3번 제5장.  그리고 여름이 와 시, 아웃소핑

[보수성 요인들에 중대한 영양 유지된 조성형 기운은 기반 유조, 그리고 무중성. 그리고 조사된 자수수: 재화 즉 세력 분쟁 재앙, 일상 경험을 토대로 자질 평가의 지식이나 무작 수식 유합은 과정이 이질로 나타나서 기도에 지대한 영구 성을 얻은 예언 가능. 보다 상세해서 말할 것은 사무직에서 전의를 전하세요.
## 부 삶(고급): 어떤 단계별 조상력 있는 소설 속 소용이량으로 바디 크릭링
...
 ... 이곳으로 진행되는 과격한 상회는 보스 인적 부력 (다리 기회), 상해상, 어제 꿈 꿀 아무 없는 이야기를 발화해서 삶

   [인성관의 자유에 지양한다 해는 아니다(주지), 전국 단계 추격.]






Here are a breakdown your the information for easy comprehension about one the famous Confucian scholar in the Korean Confucian thought is the wisdom & ability the way one does good &  achieving great things.
   

In [120]:
text= "세종대왕이 누구야?"

In [ ]:
chat_text = tokenizer.apply_chat_template (
    [
        {"role": "user", "content": text},
    ],
    tokenize= False,
    # 모델이 답변할 수 있게 
    add_generation_prompt = True,
)

In [128]:
print(chat_text)

<bos><start_of_turn>user
세종대왕이 누구야?<end_of_turn>
<start_of_turn>model



In [129]:
input_ids = tokenizer(chat_text, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids = input_ids.input_ids,
    temperature = 2.0,
    do_sample = True,
)

In [134]:
print(tokenizer.decode(output,skip_special_tokens=False)[0])

<bos><bos><start_of_turn>user
세종대왕이 누구야?<end_of_turn>
<start_of_turn>model
제임스 프랭스의 정렬자 제가 세종에게 잘 부른 이야기를 다시 말하시시?:))  이것은 지금 왜 말하며 제목의 개별 명함이나 조명을 이용하십시오!! 
무소송에 영향받고 주입할 것은 문제였잖아.  


  <end_of_turn>


In [135]:
input_ids = tokenizer(chat_text, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids = input_ids.input_ids,
    temperature = 2.0,
    do_sample = True,
    top_k = 15,
)

In [136]:
print(tokenizer.decode(output,skip_special_tokens=False)[0])

<bos><bos><start_of_turn>user
세종대왕이 누구야?<end_of_turn>
<start_of_turn>model
세세온왕 (세종대왕)** (조선 시대, 태조의 조부모에 비해 매우 인정되지 않는 왕)는 조 이사가 주도했다,  다이가 이를 통해 국권을 독립시켰다, 고위 직계인의 영향과 부하들의 도덕으로 이어졌다


## 

 세종대왕 ( 세종왕  ) 에 대해 더 자세하게 알려 드릴 게 있습니다.:   세 종 이후 50년 간 국가는  고유하고 강하게  가진 왕이라는 평가.  

**세종 대왕 (태조 제1왕, 1572~154,01)**는 역사에 탁명하며  국왕의 지능력에서 자아를 보여주었다.: 


**1. 교육과 지식:**   
세종대왕의  교육 기반이 중요한 요소가 이 시스템으로 전래된다.. 전통 교육과 과도를 통해  조상세종과  세 종에 영향을 주었고,   대왕은  가족과 함께 국가를  조선을 유지하는 데 탁월한 경험의  영감을 제공한다. 그러는 전통 과정으로,  조선에 영감을 더할 것입니다.: 



**2. 정책**   전설,   세금의 부활이  주요한 핵심 과제이자, 고귀함도 뛰어날 만큼 엄연하고 지도자로 이르기에 전형으로 이끌었다.

**결의에 도출된 결과를 분석합니다.:**  정책은 조직적인 측면, 예산과 비상계약에 영속적으로 접근하였으며 전국적인 인식을 고안했다.  결의 긍정적 결과는 뛰어 훌륭하며, 국화과 전시적인 관습  전시적인 영감을 더한다.



**3. 문화**  세종이 주관적인 조성과 전형화를 보인다 .  조선 역경은 정책의 영향이 강하게 보이지만 세종대왕의  문화적인 영감과 지능에 큰 의견이 있다.: 







**핵심 포인트의 세차인 것들을 살릴 것입니다:**



-  세종대왕의 교육 

- 세종대왕의 정책과 역학   세종대원으로 전국적인 영양과 역학이 조성 

 


 궁금하니 뭘 알면 편히 질문해 주세요!!<end_of_turn>


In [137]:
input_ids = tokenizer(chat_text, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids = input_ids.input_ids,
    temperature = 2.0,
    do_sample = True,
    top_p = 0.9,
)

In [139]:
print(tokenizer.decode(output,skip_special_tokens=False)[0])

<bos><bos><start_of_turn>user
세종대왕이 누구야?<end_of_turn>
<start_of_turn>model
## 세종대왕이라고 합니다 😄 역대 가장 장세운 왕이세요!

한국 역사에 세종대왕은 유명한 덕분에 대한민국의  기초를 위신하게 하세요! 그는 탁월한 통찰력 있는 유교 정략이기때문과 훌륭한 통찰력으로 왕국에 기여했습니다

*** 

 ✨**주요 정책 활동: 정세 관객과 지지의 능수와**


* **의제 통치 :** 기억하기  한, 정성껏 살고 행군하여  이해와 용기를 실시, 다정의왕실까지  조선 멸망 희로
* 다방면  개선  
   * **신군: "금 돈도 많고 노인도 있네!":** 강력한 정 military 군의  세진력화와 영유
   *  행정 구비 정부를 만들었고 이들은 전부 공개화되었습니다. 이것이 '고시의 통찰'이었기에 국가 의지  조정되었습니다.


**세종대왕의 대리:** 


* 조약에 부여와 조사의 재료는 한양까지 이 희미시  현재 세상 역설
* 기법적인 융해, 정리: 현실 지혜 고려하여 조세를 완경하여 조세 정책이라고



이외에도  어떻거나 특별하게 조심하세요

* 역정 개개, 개인  수집, 통화 개개도 가능 하루에 영제



<end_of_turn>


In [ ]:
input_ids = tokenizer(chat_text, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids = input_ids.input_ids,
    temperature = 2.0,
    do_sample = True,
    top_p = 0.9,
    top_k = 15,
)
print(tokenizer.decode(output,skip_special_tokens=False)[0])

In [141]:
print(tokenizer.decode(output,skip_special_tokens=False)[0])

<bos><bos><start_of_turn>user
세종대왕이 누구야?<end_of_turn>
<start_of_turn>model
세종대왕(세진나라)은 **조선시대의 왕으로 (1380-1450)** 에 재봉해 한국 역사에서 매우 중요한 인물입니다. 그는 조선 시대 최초의 영장 왕을 통치하고 세진(Joseon) 왕국의 건강한 기초가 되었습니다. 

## 세종 대왕: 전문성, 헌경, 문화적 업적이 넘칩으로 남겼어요. 

그는 국무원을 통하여 조선을 혁신했습니다. 15대 왕실에 있었고 그의 업적에 대해 좀 더 알려드리면 다음과 같은 부분이 있습니다.

* **"한자를 통한 문학, 음법 등 교육 개혁"**: 그는 문장의 표준화에 도움의 새로운 언문학으로 이념이 생략되었다. 이것은 국민의 훈련과 발전으로 이루어진 교육의 기초를 닦고 한국의 학문과 문화 발전을 선행해 냈다고 볼 수 있어요.
* **신기술 개발과 헌책 발간**: 세종대왕은 기본을 이전해 기술과 탐구로 삶에 도움과 진로를 탄탄하게 만들도록 노력하는 사람으로 유명해, 특유의 긍정적인 변형이 있을 수 있습니다.
* **중국과의 친밀한 조건 구제 및 통신의 전개에 관해 연구하는 능률을 보여주었습니다. **

## 세종대왕과 국민들의 연동 강조
  * 그의 이름을 막어, "제조인은 나만 이 훌륭한 왕을 보유한 희열이었다.", "세종 대왕, 조선의 신화, 세종과 조선은 모두 아름다움을 보였습니다." 와 같이 부탁할 가능성이 있어요.
 
 흥미로운 점은  조선이 이렇다고 해도 정량화가 부담스러우네요, 정확할 수밖에 닿지 않아요! 😊 


어떤 부분을 더 원해도 궁금해서 답변해 주셔서 고맙다는 느낌입니다 😄 <end_of_turn>


In [147]:
input_ids = tokenizer(chat_text, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids = input_ids.input_ids,
    temperature = 2.0,
    do_sample = True,
    min_p = 0.2,
    top_k =15,
    top_p = 0.7
    
)
print(tokenizer.decode(output,skip_special_tokens=True)[0])

user
세종대왕이 누구야?
model
세종대왕(1392년 12월 8일 ~ 1450년 3월 17일)은 조선시대의 세력을 축소하여 한국의 왕국을 재구축하기 위해 노력하는 인물입니다. 그의 역사는 아래와 같습니다. 

* **세종 대왕은 조선 왕조의 3번째 황제이자, 세종대왕의 역대 최초의 국왕은 조선시대 최고의 왕으로, 왕권을 넓히는 데 기여한 왕이다.** 그는 왕의 명령에 따라 왕족을 지도했고, 정치적이고 문화적으로 큰 영향을 끼쳤습니다.
* **1400년대 초, 그는 한국의 국제적인 위치를 재정립하는 데 탁월한 역할을 했습니다.** 그는 외교 및 무력력으로 외국과의 관계를 강화했고, 한국이 국제적인 위기에 빠져나갈 가능성을 낮추기 위한 전술을 세워 숙명으로 여겼던 1392년, 조선 왕조의 첫 번째 왕이자 세종대왕이 되었다.
* **그는 또한 한글을 발명하고 국립과 학자를 설립하는 데 주역을 하였다.** 한글을 발명했던 것은 한국의 문명에 대한 영향력을 확장하고, 교육에 중요성을 부여하여 세종대왕이 국문을 강조했다.


**세종대왕은 한국 역사에서 가장 영향력 있는 왕 중 한 명으로, 그의 통치 기간 동안 한국은 쇠퇴한 시대를 이끌어 나갈 수 있었다.  그의 헌신적인 정책은 한국을 긍정적으로 변화시키는 데 매우 큰 기여가 됐다. ** 





In [148]:
input_ids = tokenizer(chat_text, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids = input_ids.input_ids,
    temperature = 2.0,
    do_sample = True,
    min_p = 0.2,
    top_k =15,
    top_p = 0.7,
    repetition_penalty = 1.2
    
)
print(tokenizer.decode(output,skip_special_tokens=True)[0])

user
세종대왕이 누구야?
model
**세종 대 왕 (1392-1450)**은 조선의 세 번째 국왕으로, 명나라에서 격려된 한국 고조계를 이루어 내기 위한 통일적인 정책을 시행했다. 그의 치세는 **'삼국가상사(三國加史)'라는 전설적 기념과 함께 대한민국 역사와 문화에 중요한 영향력을 행정 개혁 및 군위 확립에도 큰 부분을 차지했던 흥미로운 사건들이 있습니다.**

### 세종 대 왕의 생애 :

* 뛰어난 문학자, 철학자, 예술인으로 평판도 높았고 유명하였으며, 특히 인문 학문과 예법 등의 연구와 실제 운영에 대해 강렬하게 관찰하여 많은 변화로 이끌었다. 


* 그는  대한민국의 발전을 위해 매우 적극적으로 노력하며 다양한 사업들을 수립하였다.



* 세 종은 한반도 사회 분단의 심각성에 주목하여 여러 국가들과 긴장 관계 없이는 지리적 접촉 증진을 추구했습니다.


## 세종대왕의 업적:

* **교통망 구축**: 도량하고 광범위함 있는 교류를 허용하기 위하여 새로운 건물 설계 기술 개발을 통해 경제 활동 증가와 더 나아가 서부 세계에 대한 상호관련 제품 공급원 연결 가능성을 향유하는데 초점을 두었지만 그러한 목표 달성에는 아직 어떤 단절이 존재합니다.  

* **군수 관리**: 장거리 이동이나 산업 방식 등을 재개발시키며 군산업 발달을 위해 최근의 기술 트렌드 반영을 시켰습니다.  현금 거래를 보다 안보 있게 하고 투쟁할 수 있도록 하는 것 또한 중심 요소였다. 




* **새로운 언론**: 일치되면서 새로운 정보 제공 시스템의 발견으로 각 지역 사람들의 참여 자체의 성장에 효율성을 높이고, 민족주의 교육 시험을 진행하면서 동료들은 균형적인 의무 회복에 맞춰 과정 완료를 돕는 것을 지원하기 위해 애매한 내용이 있어요!
    

결론부터 말해 드릴까요. 세종대왕의 영광스러운 정권은 단순한 국왕의 권리를 의미한다. 그의 업적들은 오늘날 우리가 살고 있는 모든 사회 구성 요소를 포함하지만 특별한 경우입니다! 








